In [230]:
"""
Constructs the DFT of the dihedral group in SageMath using knowledge of the representation theory.

TO-DO: work over finite field.

notes: # Gershgorin's theorem did not provide good bounds for the eigenvalues.

""";

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
n = 3; print("n =", n)
USE_FINITE_FIELD = False
q = 11  # prime power; ignored when USE_FINITE_FIELD = False

n = 4


In [232]:
# ── Build the coefficient ring and choose omega ─────────────────────────────────
if USE_FINITE_FIELD:
    # Find the smallest k such that n | (q^k - 1),
    # i.e. the multiplicative order of q mod n.
    k = Zmod(n)(q).multiplicative_order()
    F = GF(q**k, 'a')
    print(f"Working in GF({q}^{k}) = GF({q**k})")

    # Find a primitive n-th root of unity in F.
    # The multiplicative group of GF(q^k) is cyclic of order q^k - 1,
    # so a generator g satisfies g^((q^k-1)/n) has order n.
    g = F.multiplicative_generator()
    omega = g**((q**k - 1) // n)
    assert omega**n == F.one(), "omega is not an n-th root of unity"
    assert omega**(n-1) != F.one() or n == 1, "omega is not primitive"
    print(f"omega = {omega}  (order {omega.multiplicative_order()})")
else:
    K.<z> = CyclotomicField(n) #cyclotomic field containing a primitive n-th root of unity
    omega = z; print(f"omega = {omega}  (primitive n-th root of unity)")

omega = z  (primitive n-th root of unity)


In [233]:
G = DihedralGroup(n); print("G =", G) #D_n, dihedral group of order 2n
r = [g for g in G if g.order() == n][0] #rotation of order n, generator
s = [g for g in G if g.order() == 2 and g != r**(n//2)][0] #flip of order 2, generator
print("r =", r)
print("s =", s)

G = Dihedral group of order 8 as a permutation group
r = (1,4,3,2)
s = (2,4)


In [234]:
# returns (0, k) if g = r^k and (1, k) if g = s*r^k
def express_in_gens(g):
    for k in range(n):
        if g == r**k:
            return (0, k)
    for k in range(n):
        if g == s * r**k:
            return (1, k)

In [235]:
# n odd, we have two 1-dim'l irreps and (n-1)/2 2-dim'l irreps
# the 1-dim's irreps are trivial and sign
# the 2-dim'l irreps are given by rotation matrices and a flip matrix
def rho_odd(k, g, omega):
    (s_exp, r_exp) = express_in_gens(g)
    if k == 0:
        return matrix([1])
    if k == -1:
        if s_exp == 0:
            return matrix([1])
        if s_exp == 1:
            return matrix([-1])
    if k >= 1:
        if s_exp == 0:
            return matrix([[omega**(k*r_exp), 0], [0, omega**(-k*r_exp)]])
        if s_exp == 1:
            return matrix([[0, omega**(k*r_exp)], [omega**(-k*r_exp), 0]])

In [236]:
def dft_matrix_odd(omega):
    assert n % 2 == 1
    rows = []
    for g in G:
        row = [rho_odd(k, g, omega).list() for k in range(-1,(n-1)//2 + 1)]
        rows.append(sum(row, []))
    return matrix(rows)

In [237]:
# for n even case
def rho_even(k, g, omega):
    (s_exp, r_exp) = express_in_gens(g)
    if k == 0:   # trivial
        return matrix([1])
    if k == -1:  # sign of rotation
        return matrix([(-1)**r_exp])
    if k == -2:  # sign of reflection
        return matrix([(-1)**s_exp])
    if k == -3:  # total sign
        return matrix([(-1)**(r_exp + s_exp)])
    if k >= 1:
        if s_exp == 0:
            return matrix([[omega**(k*r_exp), 0], [0, omega**(-k*r_exp)]])
        if s_exp == 1:
            return matrix([[0, omega**(k*r_exp)], [omega**(-k*r_exp), 0]])

In [238]:
# form the DFT matrix for n even
def dft_matrix_even(omega):
    assert n % 2 == 0
    rows = []
    for g in G:
        row = [rho_even(k, g, omega).list() for k in range(-3, n//2)]
        rows.append(sum(row, []))
    return matrix(rows)

In [239]:
DFT_matrix = dft_matrix_odd(omega) if n % 2 == 1 else dft_matrix_even(omega); print(DFT_matrix)

[ 1  1  1  1  1  0  0  1]
[ 1  1  1  1 -1  0  0 -1]
[-1  1 -1  1  z  0  0 -z]
[-1  1 -1  1 -z  0  0  z]
[-1 -1  1  1  0  1  1  0]
[-1 -1  1  1  0 -1 -1  0]
[ 1 -1 -1  1  0  z -z  0]
[ 1 -1 -1  1  0 -z  z  0]


In [240]:
f = DFT_matrix.charpoly(); print(f)

x^8 + (z - 1)*x^7 + (-2*z - 4)*x^6 + 8*x^5 + (-8*z - 8)*x^4 + (-32*z - 32)*x^3 + 128*x^2 + 256*z*x - 1024


In [241]:
if USE_FINITE_FIELD:
    L.<a> = f.splitting_field(); L

In [242]:
if USE_FINITE_FIELD:
    eigenvalues = f.roots(L, multiplicities=False); eigenvalues

In [243]:
def frobenius_orbit(alpha, p):
    orbit = []
    seen = set()
    
    x = alpha
    while x not in seen:
        seen.add(x)
        orbit.append(x)
        x = x^p
    
    return orbit

In [244]:
def frobenius_orbits(eigenvalues, p):
    orbits = []
    seen = set()
    
    for lam in eigenvalues:
        if lam not in seen:
            orb = frobenius_orbit(lam, p)
            orbits.append(orb)
            seen.update(orb)
    
    return orbits

In [245]:
def discrete_log_Fq(x, alpha=None):
    F = x.parent()
    if x == 0:
        raise ValueError("Log undefined for 0")
    if alpha is None:
        alpha = F.multiplicative_generator()
    return x.log(alpha)

In [246]:
frobenius_orbits(eigenvalues, p)

[[8, 6, 7, 2],
 [3*a^7 + 4*a^6 + 6*a^4 + 5*a^3 + 6*a^2 + 4*a + 2,
  3*a^8 + 7*a^7 + 7*a^6 + 10*a^5 + 3*a^4 + 8*a^3 + 6*a^2 + 5*a + 2,
  7*a^8 + 5*a^7 + 7*a^6 + 8*a^4 + 5*a^3 + 4*a^2 + 3*a + 6,
  9*a^8 + 4*a^7 + 9*a^6 + a^5 + a^4 + 3*a^3 + 5*a^2 + 2*a + 6,
  9*a^8 + 10*a^6 + 2*a^4 + 8*a^3 + 4*a^2 + 3*a + 1,
  2*a^8 + 8*a^7 + 9*a^6 + 10*a^5 + 10*a^4 + 4*a^3 + 10*a^2 + 4*a + 6,
  7*a^8 + 10*a^7 + 4*a^6 + 10*a^5 + 4*a^4 + 10*a^3 + 9*a^2 + 3*a + 9,
  2*a^8 + 5*a^7 + 3*a^6 + 8*a^5 + 8*a^4 + 2*a^3 + 5*a^2 + 6*a + 7,
  4*a^8 + 5*a^7 + 5*a^6 + 6*a^5 + 8*a^4 + 6*a^3 + 9*a^2 + 4,
  2*a^8 + 8*a^7 + 7*a^6 + a^5 + a^4 + 3*a^2 + 3*a + 2,
  4*a^8 + a^7 + 7*a^6 + 5*a^5 + a^4 + 3*a^3 + 10*a^2 + 10*a + 10,
  2*a^8 + 5*a^7 + 6*a^6 + 5*a^5 + 3*a^4 + 6*a^3 + 5*a^2 + 7*a + 9,
  8*a^8 + 2*a^7 + 2*a^6 + 6*a^2 + 9*a + 1,
  4*a^8 + 10*a^6 + 5*a^4 + 5*a^3 + 3*a^2 + 4*a + 10,
  6*a^8 + 3*a^7 + 8*a^6 + 7*a^5 + a^3 + 3*a^2 + 7*a,
  a^8 + 3*a^7 + 2*a^6 + 3*a^4 + 9*a^3 + 7*a^2 + 5*a + 8,
  10*a^8 + 6*a^7 + 8*a^6 + 3*a

In [247]:
dlogs = [discrete_log_Fq(x, alpha=None)/(L.order()-1) for x in eigenvalues]; dlogs

[3/10,
 87887147/168424835,
 58202932/168424835,
 134957747/168424835,
 90957167/168424835,
 137136537/168424835,
 124634442/168424835,
 23580182/168424835,
 161103227/168424835,
 158404662/168424835]

In [248]:
if USE_FINITE_FIELD:DFT_matrix.eigenvalues()

In [249]:
# can normalize by 1/sqrt(D) to get a unitary matrix
if not USE_FINITE_FIELD:
    print(DFT_matrix.conjugate_transpose() * DFT_matrix)

[8 0 0 0 0 0 0 0]
[0 8 0 0 0 0 0 0]
[0 0 8 0 0 0 0 0]
[0 0 0 8 0 0 0 0]
[0 0 0 0 4 0 0 0]
[0 0 0 0 0 4 0 0]
[0 0 0 0 0 0 4 0]
[0 0 0 0 0 0 0 4]


In [250]:
# form norm polynomial by acting on coefficients of characteristic polynomial by the Galois group
def norm_poly(f, n):

    def sigma(i):
        phi = K.hom([z**i])
        return f.map_coefficients(phi)
    
    units = [i for i in range(1, n) if gcd(i, n) == 1]  # (Z/nZ)^×
    result = prod(sigma(i) for i in units)
    return result.change_ring(QQ)

In [251]:
f = DFT_matrix.charpoly(); f

x^8 + (z - 1)*x^7 + (-2*z - 4)*x^6 + 8*x^5 + (-8*z - 8)*x^4 + (-32*z - 32)*x^3 + 128*x^2 + 256*z*x - 1024

In [252]:
Nf = norm_poly(f, n); Nf

x^16 - 2*x^15 - 6*x^14 + 20*x^13 - 12*x^12 - 128*x^11 + 416*x^10 - 2944*x^8 + 4096*x^7 + 8192*x^6 - 28672*x^5 + 16384*x^4 + 65536*x^3 - 196608*x^2 + 1048576

In [253]:
Nf.is_irreducible()

True

In [255]:
Nf.galois_group()

NotImplementedError: unable to start kash because the command 'kash3 -b -c -d  ' failed: The command was not found or was not executable: kash3.

Sorry, computation of Galois groups of fields of degree bigger than 11 is not yet implemented.  Try installing the optional free (closed source) KASH software, which supports degrees up to 21, or use algorithm='magma' if you have magma.

In [ ]:
G = TransitiveGroup(12, 299)
print(G.order())
print(G.is_solvable())
print(G.is_primitive())
print(G.structure_description())

1036800
False
False
(A6 x A6) : D4
